# SDK prunding


### Local SDK Path


In [8]:
SDK_PATH = '/root/autodl-tmp/revitdocs/Samples'

## Get ReadMe Doc

In [7]:
import os
import json
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai as genai

def read_readme_doc(path: str) -> dict:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # make rtf doc -> text doc 
    plain_text_content = rtf_to_text(content)

    # The f-string prompt with escaped curly braces in the JSON example
    prompt_sdk_select = f"""
        # ROLE
        You are an AI assistant specializing in codebase analysis, an expert at extracting structured data from technical documentation.

        # GOAL
        Your goal is to accurately parse the provided ReadMe file to extract key identifiers for code. This output will be used programmatically by an automated code retrieval and analysis system, so the accuracy and format of your response are critical.

        # INSTRUCTIONS
        1.  Carefully analyze the text provided within the `<ReadMeContent>` tags.
        2.  Extract the following three categories of information:
            - `target_files`: A list of all project source filenames (e.g., `.cs` files) explicitly mentioned in the text.
            - `key_classes_and_methods`: A list of the names of custom classes or methods created *within* the project that are identified as being responsible for core functionality.
            - `mentioned_apis`: A list of key API classes from external frameworks or libraries (e.g., `Autodesk.Revit.DB.View`) that are explicitly listed in the text.
        3.  Format your output as a single, strict JSON object.
        4.  If no information is found for a specific field, its value must be an empty list (`[]`). Do not omit the key from the JSON object.
        5.  Your final response **MUST** contain *only* the raw JSON object, without any explanatory text, markdown code blocks, or other conversational filler.

        # EXAMPLE
        <ExampleReadMe>
        Summary: This tool is in the file `Processor.cs`. The core logic is handled by the `DataParser` class, which uses the `Autodesk.Revit.DB.Transaction` API.
        </ExampleReadMe>
        <ExampleJSONOutput>
        {{
        "target_files": ["Processor.cs"],
        "key_classes_and_methods": ["DataParser"],
        "mentioned_apis": ["Autodesk.Revit.DB.Transaction"]
        }}
        </ExampleJSONOutput>

     
    """

    # initialize openai
    load_dotenv(dotenv_path='/root/autodl-tmp/python_revit_train/gemini_api.env')
    # Corrected environment variable name for consistency
    deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
    if not deepseek_api_key:
        raise ValueError("Error: DEEPSEEK_API_KEY environment variable not set.")
        
    #genai.configure(api_key=gemini_api_key)

    #model = genai.GenerativeModel('gemini-1.5-flash-latest')

    #response = model.generate_content(prompt_sdk_select)

    # Create Response
    client = OpenAI(api_key=deepseek_api_key , base_url="https://api.deepseek.com")

    response = client.chat.completions.create(
    model="deepseek-chat",
        messages=[
            {"role": "system", "content": prompt_sdk_select},  
            
            {"role": "user", "content": f"{plain_text_content}"}
        ],
        stream = False
    )

    # response = gemini_model.generate_content(query_llm)

    # print(f"query: {query_llm}")
    print("Response from DeepSeek:")
    print(response.choices[0].message.content)



    cleaned_json_string = response.choices[0].message.content.strip().replace("```json", "").replace("```", "").strip()

    return json.loads(cleaned_json_string)


if __name__ == "__main__":
    # Ensure the striprtf library is installed: pip install striprtf
    target_content = read_readme_doc('/root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf')
    print(json.dumps(target_content, indent=2, ensure_ascii=False))

Response from DeepSeek:
{
  "target_files": ["AllViews.cs", "AllViewsForm.cs"],
  "key_classes_and_methods": ["Command", "ViewsMgr", "AllViewsForm"],
  "mentioned_apis": ["Autodesk.Revit.DB.View", "Autodesk.Revit.DB.ViewSet", "Autodesk.Revit.Creation.Document.NewViewSheet"]
}
{
  "target_files": [
    "AllViews.cs",
    "AllViewsForm.cs"
  ],
  "key_classes_and_methods": [
    "Command",
    "ViewsMgr",
    "AllViewsForm"
  ],
  "mentioned_apis": [
    "Autodesk.Revit.DB.View",
    "Autodesk.Revit.DB.ViewSet",
    "Autodesk.Revit.Creation.Document.NewViewSheet"
  ]
}


## Get This Project Most Important Code Setences

#### revit sdk sampl code

In [6]:
csharp_code_revit = """
//
// (C) Copyright 2003-2023 by Autodesk, Inc. All rights reserved.
//
// Permission to use, copy, modify, and distribute this software in
// object code form for any purpose and without fee is hereby granted
// provided that the above copyright notice appears in all copies and
// that both that copyright notice and the limited warranty and
// restricted rights notice below appear in all supporting
// documentation.

//
// AUTODESK PROVIDES THIS PROGRAM 'AS IS' AND WITH ALL ITS FAULTS.
// AUTODESK SPECIFICALLY DISCLAIMS ANY IMPLIED WARRANTY OF
// MERCHANTABILITY OR FITNESS FOR A PARTICULAR USE. AUTODESK, INC.
// DOES NOT WARRANT THAT THE OPERATION OF THE PROGRAM WILL BE
// UNINTERRUPTED OR ERROR FREE.
//
// Use, duplication, or disclosure by the U.S. Government is subject to
// restrictions set forth in FAR 52.227-19 (Commercial Computer
// Software - Restricted Rights) and DFAR 252.227-7013(c)(1)(ii)
// (Rights in Technical Data and Computer Software), as applicable. 

using System;
using System.Windows.Forms;

using Autodesk.Revit.UI;

using TaskDialog = Autodesk.Revit.UI.TaskDialog;

namespace APIAppStartup
{
   [Autodesk.Revit.Attributes.Transaction(Autodesk.Revit.Attributes.TransactionMode.Manual)]
   [Autodesk.Revit.Attributes.Regeneration(Autodesk.Revit.Attributes.RegenerationOption.Manual)]
   [Autodesk.Revit.Attributes.Journaling(Autodesk.Revit.Attributes.JournalingMode.NoCommandData)]
   public class AppSample : IExternalApplication
   {
      #region IExternalApplication Members

      public Autodesk.Revit.UI.Result OnShutdown(UIControlledApplication application)
      {
         TaskDialog.Show("Revit", "Quit External Application!");
         return Autodesk.Revit.UI.Result.Succeeded;
      }

       public Autodesk.Revit.UI.Result OnStartup(UIControlledApplication application)
      {
         String version = application.ControlledApplication.VersionName;

         //display splash window for 10 seconds
         SplashWindow.StartSplash();
         SplashWindow.ShowVersion(version);
         System.Threading.Thread.Sleep(10000);
         SplashWindow.StopSplash();

         return Autodesk.Revit.UI.Result.Succeeded;
      }

      #endregion
   }
}


"""

### tree-sitter


In [5]:
from tree_sitter import Language , Parser , Query ,Node
import tree_sitter_c_sharp
import collections

def ini_query(content : str) :
    CSHARP_LANGUAGE =  Language(tree_sitter_c_sharp.language())
    # this is a easy code to get value
    csharp_code_for_query = """
    public class Calculator
    {
        public int Add(int x, int y) => x + y;
        private static string GetWelcomeMessage() => "Welcome!";
    }
    """

    cpp_parser = Parser(CSHARP_LANGUAGE)


    tree = cpp_parser.parse(bytes(content, "utf8"))
    root_node = tree.root_node
    return CSHARP_LANGUAGE , root_node

def get_details_query(class_name : str , root_node : Node , lang : Language) -> list:
    """
    input class_name that get all method context 
    
    """
    # 定义一个查询字符串 | get a main query to get all code method in class
    # - 查找所有 method_declaration 节点 | find all method_declaration block
    # - 在该节点下，捕获返回类型 (predefined_type 或 identifier) 并命名为 @return.type in this block , get return type and named to @return.type
    # - 捕获方法名 (identifier) 并命名为 @method.name | in this block get method name and named to @method.name
    # https://tree-sitter.github.io/tree-sitter/7-playground.html this is a online website that to check query structure
    query_string = f"""
        (compilation_unit
            (namespace_declaration
                body: (declaration_list
                    (class_declaration
                        name: (identifier) @class.name
                        (base_list) @base.list.name?
                        (#any-of? @class.name {class_name})
                        body: (declaration_list
                        (method_declaration) @method.node
                        )
                    )
                )
            )
        )
    """
    # print(f'first query str : {query_string}')
    # get the method query result 
    query = Query(lang, query_string)

    # 对语法树执行查询
    # captures = query.captures(root_node)
    # print(captures)

    # use matches to get all code and return a tuple[int , dic[int , list[node]]]
    matches = query.matches(root_node) # return a tuple
    final_methods_list = []
    for match in matches:
        # the second query to split the parameters , this can get muti-parameter in method 
        query_sub_string = """
        (
                method_declaration
                returns: (_) @return.type
                name: (identifier) @method.name
                parameters: (parameter_list
                    (parameter) @param.complete
                )*
                body : (_) @method.body
        )
        """
        # get the target node which has method type and name 
        # print('start sub query ')
        values = match[1]
        value_node = values['method.node'][0]
        get_details_query = Query(lang, query_sub_string)
        detail_captures = get_details_query.captures(value_node) # return a dictionary

    
        # define a struct : name , return type and params
        method_details = {
                "name": "",
                "return_type": "void",
                "params": []  , # this is a params group
                "body" : ""
            }
        
                
        # group to params
        param_nodes = []
        # details_captures is a dictionary so need use items
        for name , node in detail_captures.items():
            if name == 'method.name':
                method_details['name'] = node[0].text.decode('utf8')
            elif name == 'return.type':
                method_details['return_type'] = node[0].text.decode('utf8')
            elif name == 'param.complete':
                # 将找到的完整参数节点添加到临时列表中
                for sub_node in node :
                    param_nodes.append(sub_node)
            elif name == 'method.body' :
                method_details['method.body'] = node[0].text.decode('utf-8')
                    
            # union the paramas value
        for param_node in param_nodes:
            method_details['params'].append(param_node.text.decode('utf8'))
                
        final_methods_list.append(method_details)
        
    # prinf
    #print(final_methods_list)
    return final_methods_list


if __name__ == "__main__" :
    configs = ini_query(csharp_code_revit)
    l = get_details_query("AppSample SplashWindow OnStartup OnShutdown" , configs[1] , configs[0])
    print(l)


NameError: name 'csharp_code_revit' is not defined

In [9]:
import os
from typing import List , Dict , Any

def get_all_files(root_path : str)  -> List[Dict[str, Any]]:
    """
    扫描一个根目录，找到所有项目文件夹，并为每个项目找到ReadMe.rtf和所有文件的路径。

    Args:
        root_path: 要扫描的根目录路径 (例如: '/root/autodl-tmp/revitdocs/Samples/')。

    Returns:
        一个项目信息列表。每个项目是一个字典，包含:
        - 'project_name': 项目文件夹的名称。
        - 'project_path': 项目文件夹的完整路径。
        - 'readme_path': 'ReadMe.rtf' 文件的完整路径 (如果找到的话，否则为 None)。
        - 'all_files': 项目中所有文件的完整路径列表。
    """
   
    if not os.path.isdir(root_path):
            print(f"❌ 错误：提供的根路径 '{root_path}' 不是一个有效的目录。")
            return []

    projects_list = []
    print(f"🚀 开始高级扫描根目录: {root_path}")

    # os.walk() 会自顶向下地遍历整个目录树
    for dirpath, dirnames, filenames in os.walk(root_path):
            
            # --- 新需求 2: 过滤VB.NET项目 ---
            # 检查当前文件夹是否包含任何 .vb 文件
            has_vb = any(f.lower().endswith('.vb') for f in filenames)
            if has_vb:
                print(f"⏭️  跳过VB.NET项目: {dirpath}")
                # 清空dirnames列表，告诉os.walk不要再深入这个目录的任何子目录
                dirnames[:] = [] 
                continue

            # --- 新需求 1: 识别C#项目 ---
            # 检查当前文件夹是否是 'CS' 文件夹，或者直接包含 .cs 文件
            is_cs_folder = os.path.basename(dirpath).lower() == 'cs'
            has_cs_files = any(f.lower().endswith('.cs') for f in filenames)

            if is_cs_folder or has_cs_files:
                print(f"🎯 发现C#源码目录: {dirpath}")

                # --- 新需求 1: 确定项目逻辑根目录和项目名称 ---
                if is_cs_folder:
                    # 如果是'CS'文件夹，则其父目录是项目的逻辑根目录
                    project_root = os.path.dirname(dirpath)
                else:
                    # 否则，当前目录就是项目的逻辑根目录
                    project_root = dirpath
                
                # 计算相对于扫描根目录的路径，并生成项目名称
                relative_path = os.path.relpath(project_root, root_path)
                project_name = relative_path.replace(os.sep, '.')
                
                print(f"   -> 项目名称: {project_name}")
                print(f"   -> 项目根目录: {project_root}")

                # --- 搜集项目信息 ---
                project_data = {
                    "project_name": project_name,
                    "project_path": project_root,
                    "readme_path": None,
                    "all_files": []
                }

                # 再次遍历项目根目录，以收集所有文件和ReadMe
                for proj_dirpath, _, proj_filenames in os.walk(project_root):
                    for proj_filename in proj_filenames:
                        full_path = os.path.join(proj_dirpath, proj_filename)
                        project_data["all_files"].append(full_path)
                        
                        if 'readme' in  proj_filename.lower():
                            project_data["readme_path"] = full_path

                if project_data["readme_path"]:
                    print(f"   -> ✅ 找到 ReadMe 文件: {project_data['readme_path']}")
                else:
                    print(f"   -> ⚠️ 警告: 在项目 '{project_name}' 中未找到 'ReadMe.rtf'。")

                projects_list.append(project_data)
                
                # 告诉os.walk不要再深入这个已识别项目的任何子目录，避免重复
                dirnames[:] = []
                
    print("\n扫描完成！")
    return projects_list



def find_file_path_in_project(project_data: Dict[str, Any], target_filename: str) -> str | None:
    """
    在一个项目的数据字典中，根据文件名查找其完整的路径。

    Args:
        project_data: 包含项目信息的字典，必须含有 'all_files' 键。
        target_filename: 您要查找的文件的名字 (例如: "AllViews.cs")。

    Returns:
        如果找到文件，则返回其完整的路径字符串；如果未找到，则返回 None。
    """
    # 检查 'all_files' 键是否存在并且是一个列表
    if 'all_files' not in project_data or not isinstance(project_data['all_files'], list):
        print("错误：'project_data' 字典中没有找到 'all_files' 列表。")
        return None

    # 遍历项目中的每一个文件的完整路径
    for full_path in project_data['all_files']:
        # 从完整路径中提取文件名
        # os.path.basename() 可以正确处理 'A/B/C.txt' -> 'C.txt'
        if os.path.basename(full_path) == target_filename:
            # 如果文件名匹配，则返回这个完整路径
            return full_path
    
    # 如果遍历完所有文件都没有找到，则返回 None
    return None




# --- 使用示例 ---
if __name__ == "__main__":
    # 请将此路径替换为您的Revit SDK Samples的根目录
    SDK_ROOT = '/root/autodl-tmp/revitdocs/Samples' 
    
    # 执行高级扫描
    found_projects = get_all_files(SDK_ROOT)
    
    if found_projects:
       for project in found_projects :
            target_content = read_readme_doc(project["readme_path"])
            if target_content :
                 print(target_content)
                 target_files = target_content.get('target_files')
                 target_class_method = target_content.get('key_classes_and_methods')
                 if target_class_method :
                    print('target class string is ========>')
                    target_class_method_str = " ".join(target_class_method)
                    print(target_class_method_str)
                 for target_file in target_files :
                    print(f'open file : {target_file}')
                    find_file = find_file_path_in_project(project , target_file)
                    print(find_file)
                    with open(find_file, 'r', encoding='utf-8-sig', errors='ignore') as f:
                        file_content = f.read()
                    print('Ini with code content')
                    configs = ini_query(content=file_content)
                    res = get_details_query(target_class_method_str , configs[1] , configs[0])
                    if res :
                        print(f'result is : {res}')
                 break


🚀 开始高级扫描根目录: /root/autodl-tmp/revitdocs/Samples
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS
   -> 项目名称: APIAppStartup
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS/ReadMe_APIAppStartup.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AllViews/CS
   -> 项目名称: AllViews
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AllViews
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS
   -> 项目名称: AnalysisVisualizationFramework.DistanceToSurfaces
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS/ReadMe_DistanceToSurfaces.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramewo

### The Full Agent Workflow

In [20]:
import json
import os
import time
from tree_sitter import Language , Parser
from dotenv import load_dotenv
from deepseek_tokenizer_v3 import deepseek_tokenizer
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def read_api_key_from_file(path: str = "/root/autodl-tmp/python_revit_train/gemini_api.env") -> str:
    """
    从指定文件读取API密钥，返回字符串（自动去除首尾空白）。
    """
    load_dotenv(path)

    key = os.getenv('DEEPSEEK_API_KEY')
    return key

def full_agent_flow( root_path : str) :
    
    readme_path = os.path.join(root_path , 'ReadMe_AllViews.rtf')
    print('=====================================')
    print('Agent Start')
    target_content = read_readme_doc(readme_path)
    if not target_content :
        print("Fail To Use This Tools : read_readme_doc")
        return None
   
    print(json.dumps(target_content, indent=2, ensure_ascii=False))
    all_class_details = []

    for class_name in target_content.get('key_classes_and_methods' , []) :
        class_found = False
        for file_name in target_content.get('target_files' , []):
            full_path = os.path.join(root_path , file_name)

            class_detail = get_details_query(class_name , configs[1] , configs[0])

            if class_detail :
                all_class_details.append(class_detail)
                class_found = True
                break

        if not class_found :
            print(f'cant find this class_name : {class_name}')

    print('\n Agent ===> Compare Data')

    comparehensive_data = {
        "project_name" : readme_path ,
        "readme_summary" : target_content ,
        "detail_code_analysis" : all_class_details
    } 

    print("Agent Done ")
    return comparehensive_data



def generatial_clean_codes(readme_summary : str , detail_code_analysis : str) :
    print('start define prompt')

    tokenizer_length = deepseek_tokenizer.get_local_tokenizer_length(detail_code_analysis)

    if tokenizer_length > 131072 :
        print("#############################")
        print(f" code length was more than max token : 131079 ")
        return None

    prompt = f"""
        # ROLE
You are an expert C# software architect and technical writer specializing in the Autodesk Revit API. You excel at identifying core logic and refactoring it into clear, concise, and educational code examples.

# GOAL
Your mission is to analyze the provided ReadMe context and potentially complex raw C# code details. You must intelligently **identify the single most relevant code block** representing the core functionality described in the ReadMe, and then synthesize it into a clean, reusable, and perfectly documented 'Golden Code Snippet' for a RAG knowledge base.

# CONTEXT
You will be given two primary pieces of information: context extracted from the project's ReadMe file and detailed code analysis results.

<ReadMeContext>
  Project Summary: {readme_summary}
</ReadMeContext>

<RawCodeDetails>
  {detail_code_analysis} 
</RawCodeDetails>
# Note: <RawCodeDetails> contain a list/JSON of multiple extracted methods , You Need To Get The Target API And Class In this Message.

# STEP-BY-STEP INSTRUCTIONS , Follow Step One By one Thinking

1.  **Analyze Goal & Context**: First, thoroughly read the `<ReadMeContext>` to understand the project's main purpose and the key APIs involved.

2.  **Identify Core Logic Block**: Examine the `<RawCodeDetails>`.
    * If it contains multiple distinct methods or code blocks, **select the single block** that most directly implements the core functionality described in the `Project Summary` and utilizes the `Key APIs Mentioned`. Prioritize methods with significant logic over simple event handlers or boilerplate.
    * If it contains a full class, focus on the method(s) within that class that perform the primary actions.
    * If it contains only one relevant block, proceed with that block.
    Let's call the selected block the **"Target Code"**.

3.  **Filter the Target Code**: Ruthlessly filter out all non-essential elements from the **Target Code**. This includes:
    * ALL UI interaction code (`MessageBox`, `TaskDialog`, control properties like `.Text` or `.Checked`).
    * ALL logging and debugging statements (`Console.WriteLine`, `Debug.WriteLine`).
    * ALL generic file I/O (unless it's the core API function).
    * ALL boilerplate from `IExternalCommand.Execute` or similar entry points. Assume a `Document` object (usually named `doc`) is readily available or passed as a parameter.

4.  **Refactor for Reusability**: Refactor the remaining core logic from the **Target Code** into a standalone, reusable method.
    * Create a clear and descriptive method signature (name, parameters, return type). The method name should reflect the specific action being performed.
    * Convert any inputs that were originally hardcoded or came from UI controls into method parameters with appropriate C# types and descriptive names (e.g., `string sheetName`, `bool isStructural`, `ElementId levelId`).
    * Replace specific, hardcoded values (like magic numbers, specific `ElementId`s, file paths, specific names) with descriptive variables, sensible defaults (e.g., `XYZ.Zero`), or pass them as parameters if they are essential inputs.
    * Ensure the complete `Transaction` pattern (`using (Transaction tx = new Transaction(doc, "...")) {{ tx.Start(); ... tx.Commit(); }}`) is present and correctly wraps any modifications to the Revit model.

5.  **Generate Documentation**: Write a comprehensive C# XML documentation comment (`/// <summary>...`) for the newly refactored method.
    * The `<summary>` must accurately describe what the *final, refactored* code does, informed by the `Project Summary` from the ReadMe.
    * Include `<param>` tags for ALL input parameters defined in the new method signature.
    * Include a `<returns>` tag if the method returns a value, explaining what it returns.

6.  **Final Output Synthesis**: Ensure the final output is a single, complete, and syntactically correct C# method block. Double-check that all filtering and refactoring rules have been applied.the {{code}} need to check closed again , the code need in this block

# OUTPUT FORMAT (Modified Section)
- Provide ONLY a single string literal representing a Python dictionary, enclosed in SINGLE quotes (`'`).
- The dictionary MUST have exactly two keys: 'summary' (string) and 'content' (string containing the C# code).
- The format MUST precisely match: `{{'summary': 'Your summary text here.', 'content': 'Your_properly_escaped_C#_code_string_here.'}}`
- **Crucially, within the 'content' string value, EVERY line of the generated C# code MUST start with `/// <summary>`.** This includes using statements, method signatures, braces, and actual code lines.
- Ensure the 'content' string adheres strictly to the PYTHON STRING LITERAL RULES defined above.
- Do NOT include markdown formatting or any text outside the final dictionary string literal.

# EXAMPLE OF CORRECT ESCAPING AND FORMATTING WITHIN 'content'
# If the C# code is:
# using System;
# public void MyMethod() {{ Console.WriteLine("Hi"); }}
# The 'content' string value in the output dictionary string literal should look like:
# '/// <summary>using System;\\n/// <summary>public void MyMethod() {{ Console.WriteLine("Hi"); }}\\n' 
# (Note: The inner curly braces for the C# method body might need double escaping \\{{ \\}} depending on how the AI handles it, or just {{}} if it treats it as literal text within the escaped string)

# PYTHON STRING LITERAL RULES FOR 'content' 
- The final output MUST be a valid Python dictionary literal represented as a string, parsable by Python's `ast.literal_eval()`.
- The value associated with the 'content' key is a STRING containing C# source code.
- Inside this 'content' string value, ensure all special characters are correctly escaped according to **STANDARD PYTHON STRING LITERAL RULES**:
    - Literal backslashes (`\`) MUST be represented as (`\\`).
    - Newline characters MUST be represented as (`\\n`).
    - Tab characters MUST be represented as (`\\t`).
    - Double quotes (`"`) within the C# code MUST be escaped as (`\\"`).
    - Single quotes (`'`) within the C# code MUST be escaped as (`\\'`) because the outer dictionary uses single quotes.


# TASK
Begin the analysis, selection, filtering, refactoring, and documentation process based on the context provided above. Generate the Golden Code Snippet.

        """
    

    # print('############## Prompt ################')
    # print(prompt)
    query  = f'Give The Clean Code Snipate'
 
    # 用法示例
    api_key = read_api_key_from_file()
    client = OpenAI(api_key=api_key , base_url="https://api.deepseek.com")
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": f"{prompt}"},  
            
            {"role": "user", "content": f"{query}"}
        ],
        stream = False
    )

    print(f"query: {query}")
    print("Final Response Query from DeepSeek:")
    print(response.choices[0].message.content)

    return response.choices[0].message.content



def process_project(project: dict):
    """
    Single Project
    """
    try:
        # 1. 解析ReadMe
        target_content = read_readme_doc(project.get("readme_path"))
        if not target_content:
            return None # 如果ReadMe解析失败，则跳过此项目
        
        target_content = read_readme_doc(project["readme_path"])
        if target_content :
            print(target_content)
            all_class_details = []
            all_data_contexts = None
            for class_name in target_content.get('key_classes_and_methods' , []) :
                class_found = False
                for file_name in target_content.get('target_files' , []):
                    all_files = project.get('all_files')
                    full_path = ''
                    for file in all_files :
                        if os.path.basename(file) == file_name :
                            full_path = file
                            break
                        
                    print(full_path)
                    if full_path : 
                        with open(full_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
                            file_content = f.read()
                    print('Ini with code content')
                    configs = ini_query(content=file_content)                
                    class_detail = get_details_query(class_name , configs[1] , configs[0])

                    if class_detail :
                        all_class_details.append(class_detail)
                        class_found = True
                        break

                if not class_found :
                    print(f'cant find this class_name : {class_name}')

                print('\n Agent ===> Compare Data')

                comparehensive_data = {
                    "readme_summary" : target_content ,
                    "detail_code_analysis" : all_class_details
                } 
                all_data_contexts = comparehensive_data

                print("Agent Done ")
                print('###### Detail Result ###########')
                print(res)
                msg  = generatial_clean_codes(json.dumps(all_data_contexts['readme_summary'] , indent=2) , json.dumps(all_data_contexts['detail_code_analysis'] , indent=2))
                if msg : 
                    return msg  

    except Exception as e:
        # 捕获处理单个项目时可能发生的任何异常，避免整个程序崩溃
        print(f"处理项目 {project.get('project_name')} 时出错: {e}")
        return None
    



if __name__ == "__main__" :

    SDK_ROOT = '/root/autodl-tmp/revitdocs/Samples' 
    clean_codes = []
    # Find All Files
    found_projects = get_all_files(SDK_ROOT)


    with ThreadPoolExecutor(max_workers=10) as executor :
        future_to_project = {executor.submit(process_project, project): project for project in found_projects}

        progress_bar = tqdm(as_completed(future_to_project), total=len(found_projects), desc="正在处理项目")
        
        for future in progress_bar:
            result = future.result()
            if result:
                clean_codes.append(result)

    print(f"\n处理完成！成功生成了 {len(clean_codes)} 个黄金代码片段。")

    """
    index = 0
    if found_projects:
       for project in found_projects :
            
            
            target_content = read_readme_doc(project["readme_path"])
            if target_content :
                print(target_content)
                all_class_details = []
                all_data_contexts = None
                for class_name in target_content.get('key_classes_and_methods' , []) :
                    class_found = False
                    for file_name in target_content.get('target_files' , []):
                        all_files = project.get('all_files')
                        full_path = ''
                        for file in all_files :
                            if os.path.basename(file) == file_name :
                                full_path = file
                                break
                        
                        print(full_path)
                        if full_path : 
                            with open(full_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
                                file_content = f.read()
                        print('Ini with code content')
                        configs = ini_query(content=file_content)                
                        class_detail = get_details_query(class_name , configs[1] , configs[0])

                        if class_detail :
                            all_class_details.append(class_detail)
                            class_found = True
                            break

                    if not class_found :
                        print(f'cant find this class_name : {class_name}')

                    print('\n Agent ===> Compare Data')

                    comparehensive_data = {
                        "readme_summary" : target_content ,
                        "detail_code_analysis" : all_class_details
                    } 
                    all_data_contexts = comparehensive_data

                    print("Agent Done ")
                print('###### Detail Result ###########')
                print(res)
                msg  = generatial_clean_codes(json.dumps(all_data_contexts['readme_summary'] , indent=2) , json.dumps(all_data_contexts['detail_code_analysis'] , indent=2))
                if msg : 
                    clean_codes.append(msg)

    """

    print(len(clean_codes))
            


🚀 开始高级扫描根目录: /root/autodl-tmp/revitdocs/Samples
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS
   -> 项目名称: APIAppStartup
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS/ReadMe_APIAppStartup.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AllViews/CS
   -> 项目名称: AllViews
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AllViews
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS
   -> 项目名称: AnalysisVisualizationFramework.DistanceToSurfaces
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS/ReadMe_DistanceToSurfaces.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramewo

正在处理项目:   0%|          | 0/183 [00:00<?, ?it/s]

Response from DeepSeek:
{
  "target_files": ["AppSample.cs", "SplashWindow.cs"],
  "key_classes_and_methods": ["AppSample", "SplashWindow", "OnStartup", "OnShutdown"],
  "mentioned_apis": ["Autodesk.Revit.UI.IExternalApplication"]
}
Response from DeepSeek:
{
  "target_files": ["AllViews.cs", "AllViewsForm.cs"],
  "key_classes_and_methods": ["Command", "ViewsMgr", "AllViewsForm"],
  "mentioned_apis": ["Autodesk.Revit.DB.View", "Autodesk.Revit.DB.ViewSet", "Autodesk.Revit.Creation.Document.NewViewSheet"]
}
Response from DeepSeek:
{
  "target_files": ["Command.cs"],
  "key_classes_and_methods": ["Command"],
  "mentioned_apis": ["Autodesk.Revit.DB.Mechanical.MechanicalSystem", "Autodesk.Revit.DB.Mechanical.Duct", "Autodesk.Revit.DB.FamilyInstance", "Autodesk.Revit.DB.Connector", "Autodesk.Revit.DB.Document"]
}
Response from DeepSeek:
{
  "target_files": ["AreaReinCurve.cs", "GeomUtil.cs", "ParameterUtil.cs"],
  "key_classes_and_methods": ["GeomUtil", "ParameterUtil"],
  "mentioned_apis": [

正在处理项目:   1%|          | 1/183 [00:13<39:35, 13.05s/it]

1
Response from DeepSeek:
{
  "target_files": ["Command.cs"],
  "key_classes_and_methods": [],
  "mentioned_apis": ["Autodesk.Revit.DB.Analysis.FieldDomainPointsByUV", "Autodesk.Revit.DB.Analysis.FieldValues", "Autodesk.Revit.DB.Analysis.SpatialFieldManager", "Autodesk.Revit.DB.BoundingBoxUV", "Autodesk.Revit.DB.Events.DocumentOpenedEventArgs", "Autodesk.Revit.DB.FaceArray", "Autodesk.Revit.DB.FilteredElementCollector", "Autodesk.Revit.DB.LocationPoint", "Autodesk.Revit.DB.UpdaterRegistry"]
}
{'target_files': ['Command.cs'], 'key_classes_and_methods': [], 'mentioned_apis': ['Autodesk.Revit.DB.Analysis.FieldDomainPointsByUV', 'Autodesk.Revit.DB.Analysis.FieldValues', 'Autodesk.Revit.DB.Analysis.SpatialFieldManager', 'Autodesk.Revit.DB.BoundingBoxUV', 'Autodesk.Revit.DB.Events.DocumentOpenedEventArgs', 'Autodesk.Revit.DB.FaceArray', 'Autodesk.Revit.DB.FilteredElementCollector', 'Autodesk.Revit.DB.LocationPoint', 'Autodesk.Revit.DB.UpdaterRegistry']}
884
916
8602


...

query: Give The Clean Code Snipate
Final Response Query from DeepSeek:
{'summary': 'This code snippet demonstrates how to create winder stairs in Revit by calculating control points from selected elements and determining maximum step counts based on stair dimensions. It handles both L-shaped and U-shaped winder configurations.', 'content': '/// <summary>/// Calculates control points and maximum step counts for winder stairs based on selected elements and stair dimensions.\n/// <summary>/// \n/// <summary>/// <param name=\"doc\">The active Revit document</param>\n/// <summary>/// <param name=\"selectedIds\">List of selected element IDs (2 for L-winder, 3 for U-winder)</param>\n/// <summary>/// <param name=\"runWidth\">The width of the stair run</param>\n/// <summary>/// <param name=\"treadDepth\">The depth of each tread</param>\n/// <summary>/// <returns>A tuple containing control points and maximum step counts</returns>\n/// <summary>public static (IList<XYZ> controlPoints, IList<uint>

正在处理项目:  99%|█████████▉| 182/183 [12:20<00:04,  4.27s/it]

query: Give The Clean Code Snipate
Final Response Query from DeepSeek:
{'summary': 'This code snippet demonstrates how to frame a wall with structural columns at specified intervals using the Revit API. It includes methods to find a family symbol, frame a wall by placing columns along its length, and place individual columns with proper rotation and level constraints.', 'content': '/// <summary>Finds a specific family symbol by name within the Revit document.\n/// <summary>/// <param name=\"rvtDoc\">The Revit document to search in.</param>\n/// <summary>/// <param name=\"familyName\">The name of the family to find.</param>\n/// <summary>/// <param name=\"symbolName\">The name of the symbol to find.</param>\n/// <summary>/// <returns>The found FamilySymbol, or null if not found.</returns>\n/// <summary>public static FamilySymbol FindFamilySymbol(Document rvtDoc, string familyName, string symbolName)\n/// <summary>{\n/// <summary>    FilteredElementCollector collector = new FilteredEleme

正在处理项目: 100%|██████████| 183/183 [12:41<00:00,  4.16s/it]

query: Give The Clean Code Snipate
Final Response Query from DeepSeek:
'{\'summary\': \'This code snippet demonstrates how to create a custom ribbon tab and panel in Revit with multiple buttons that have contextual help, custom icons, and availability controls. It shows different types of contextual help including wiki help, URL links, and CHM file help.\', \'content\': \'/// <summary>using Autodesk.Revit.UI;\n/// <summary>using System;\n/// <summary>using System.IO;\n/// <summary>using System.Windows;\n/// <summary>using System.Windows.Interop;\n/// <summary>using System.Windows.Media.Imaging;\n/// <summary> \n/// <summary>/// <summary>Creates a custom ribbon tab with multiple buttons demonstrating different contextual help types and availability controls.</summary>\n/// <summary>/// <param name=\"application\">The UIControlledApplication instance</param>\n/// <summary>/// <param name=\"addinAssemblyPath\">The path to the add-in assembly</param>\n/// <summary>public static void Create

### Sql Collection

In [ ]:
from sqlite3 import connect, Error
import ast
from tqdm import tqdm

OUT_DIR = '/root/autodl-tmp/python_revit_train'
class revitcollection:
    def __init__(self):
        self.output_dir = OUT_DIR + '/revit_sdk_collection'
        self.api_data = []

        os.makedirs(self.output_dir, exist_ok=True)  # 确保输出目录存在

    def connect_db(self):
        """
        连接到SQLite数据库
        """
        try:
            conn = connect(self.output_dir + '/revit_sdk.db')
            return conn
        except Error as e:
            print(f"数据库连接错误: {e}")
            return None
        
    def create_table(self, conn):
        """
        创建存储API信息的表
        """
        try:
            cursor = conn.cursor()
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS sdk_info (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    title TEXT,
                    content TEXT
                )
            ''')
            conn.commit()
        except Error as e:
            print(f"创建表错误: {e}")
    
    def insert_api_info(self, conn, title, content):
        """
        插入API信息到数据库
        """
        try:
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO sdk_info (title, content)
                VALUES (?, ?)
            ''', (title, content))
            conn.commit()
        except Error as e:
            print(f"插入数据错误: {e}")

    def process_by_contents(self) :
        """
        通过内容处理
        """
        conn = self.connect_db()
        if conn is None:
            return
        self.create_table(conn)
        for content in tqdm(self.api_data, desc="Inserting to DB"):
            title = content.get('title', '')
            body = content.get('content', '')
            if title and body:
                self.insert_api_info(conn, title, body)
        conn.close()
        print("所有API信息已存储到数据库中")
    
    def process_files(self, codes):
        """
        处理所有SDK文件并存储到数据库
        """
        conn = self.connect_db()
        if conn is None:
            return
        
        self.create_table(conn)
        
        
        for code_date in tqdm(codes, desc="Processing SDK files"):
                if code_date:
                    # print(code_date)
                    
                    code_json = ast.literal_eval(code_date)
                    self.insert_api_info(conn, code_json.get('summary'), code_json.get('content'))
        
        conn.close()
        print("所有API信息已存储到数据库中") 





In [18]:
sdk_collection =  revitcollection()
print(clean_codes[4])
sdk_collection.process_files(clean_codes)
db_path = OUT_DIR + '/revit_sdk_collection/revit_sdk.db'
conn = connect(db_path)
cursor = conn.cursor()

cursor.execute("SELECT id, title, content FROM sdk_info")
rows = cursor.fetchall()

for row in rows:
    print(f"ID: {row[0]}\nTitle: {row[1]}\nContent: {row[2]}\n{'='*40}")

conn.close()


{
'summary' : 'Creates a new view sheet in Revit with the specified name and places selected views on it',
'content' : '/// <summary>\n/// Creates a new view sheet in the Revit document and places the specified views on it.\n/// This method handles the transaction for creating the sheet and arranging the views.\n/// </summary>\n/// <param name=\"doc\">The Revit document where the sheet will be created</param>\n/// <param name=\"sheetName\">The name for the new sheet</param>\n/// <param name=\"viewsToPlace\">Collection of views to be placed on the sheet</param>\n/// <returns>The newly created ViewSheet object</returns>\npublic static ViewSheet CreateSheetWithViews(Document doc, string sheetName, ICollection<View> viewsToPlace)\n{\n    using (Transaction tx = new Transaction(doc, \"Create Sheet with Views\"))\n    {\n        tx.Start();\n        \n        ViewSheet newSheet = ViewSheet.Create(doc, ElementId.InvalidElementId);\n        \n        if (!string.IsNullOrEmpty(sheetName))\n    

Processing SDK files:   3%|▎         | 5/154 [00:00<00:00, 883.35it/s]


SyntaxError: '{' was never closed (<unknown>, line 1)